# Evaluation on Unseen Test Dataset

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import joblib
import json
from sklearn.metrics import classification_report

from scripts.config import *
from scripts.models import create_stage1_detector, create_stage2_classifier

In [2]:
df_test = pd.read_csv(PROCESSED_DIR / 'test_windowed.csv')

print(f"Processed test set shape: {df_test.shape}")

Processed test set shape: (824, 42)


In [3]:
normal_windows = pd.read_csv(PROCESSED_DIR / 'normal_windowed.csv')
eval_set = pd.read_csv(PROCESSED_DIR / 'eval_results.csv', index_col=0)
feature_cols = [col for col in normal_windows.columns if col not in ['window_id', 'window_attack']]

print("Setting up Autoencoder and SVM...")
autoencoder = create_stage1_detector('autoencoder')
autoencoder.fit(normal_windows[feature_cols].values)

ae_flags_eval = autoencoder.predict(eval_set[feature_cols].values)
flagged_eval = eval_set[ae_flags_eval == 1].copy()

svm = create_stage2_classifier('svm')
svm.fit(flagged_eval[feature_cols], flagged_eval['window_attack'].values, scaler=autoencoder.scaler)

Setting up Autoencoder and SVM...


d:\Sarthak\Projects\network-traffic-anomaly-analysis\venv\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(


## 2. Two-Stage Pipeline Evaluation

In [4]:
X_test = df_test[feature_cols]
y_test_true = df_test['window_attack'].values

s1_preds = autoencoder.predict(X_test.values)
s1_hits = np.sum((s1_preds == 1) & (y_test_true == 1))
s1_fps = np.sum((s1_preds == 1) & (y_test_true == 0))
actual_attacks = np.sum(y_test_true == 1)

final_preds = np.zeros(len(df_test))
if np.any(s1_preds == 1):
    final_preds[s1_preds == 1] = svm.predict(X_test[s1_preds == 1].values)

s2_hits = np.sum((final_preds == 1) & (y_test_true == 1))
s2_fps = np.sum((final_preds == 1) & (y_test_true == 0))
fp_reduced = s1_fps - s2_fps

print(f"Total Windows Analyzed:   {len(df_test)}")
print(f"Actual Attacks in Data:   {actual_attacks}")

print(f"\nStage 1 (Autoencoder Detection):")
print(f"  - Detected Anomalies:   {int(np.sum(s1_preds))}")
print(f"  - True Attacks Caught:  {int(s1_hits)} / {actual_attacks} (Recall: {s1_hits/actual_attacks:.2%})")
print(f"  - False Positives:      {int(s1_fps)}")

print(f"\nStage 2 (SVM FP-Reduction):")
print(f"  - False Positives Blocked: {int(fp_reduced)}")
print(f"  - Attacks Confirmed:       {int(s2_hits)}")
print(f"  - Final False Alarms:      {int(s2_fps)}")

print("\nDetailed Classification Report (Final Output):")
print(classification_report(y_test_true, final_preds))

results_df = df_test.copy()
results_df['final_prediction'] = final_preds
results_df.to_csv(PROCESSED_DIR / 'final_test_evaluation_results.csv', index=False)

Total Windows Analyzed:   824
Actual Attacks in Data:   455

Stage 1 (Autoencoder Detection):
  - Detected Anomalies:   594
  - True Attacks Caught:  447 / 455 (Recall: 98.24%)
  - False Positives:      147

Stage 2 (SVM FP-Reduction):
  - False Positives Blocked: 147
  - Attacks Confirmed:       391
  - Final False Alarms:      0

Detailed Classification Report (Final Output):
              precision    recall  f1-score   support

           0       0.85      1.00      0.92       369
           1       1.00      0.86      0.92       455

    accuracy                           0.92       824
   macro avg       0.93      0.93      0.92       824
weighted avg       0.93      0.92      0.92       824



In [5]:
autoencoder.save('autoencoder_stage1')
svm.save('svm_stage2')

Model and scaler saved to D:\Sarthak\Projects\network-traffic-anomaly-analysis\models
Model saved to D:\Sarthak\Projects\network-traffic-anomaly-analysis\models
